## Introduction
> Add blockquote



### 1. Environmental Set up

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

### 2. Dataset preparation

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

### 4. Federated Learning Simulation

In [ ]:
num_clients = 5
client_data_size = len(train_dataset) // num_clients

clients = []

for i in range(num_clients):
    start = i * client_data_size
    end = start + client_data_size
    clients.append(Subset(train_dataset, range(start, end)))

### 5. Global Model

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

6. Local Client Training

In [ ]:
def train_local(model, dataset):

    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    optimizer = optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    model.train()

    for data, target in loader:
        optimizer.zero_grad()

        output = model(data)
        loss = loss_fn(output, target)

        loss.backward()
        optimizer.step()

    return model.state_dict()

7. Federated Averaging(FedAvg)

In [ ]:
def federated_average(weights):

    avg_weights = {}

    for key in weights[0].keys():
        avg_weights[key] = torch.mean(
            torch.stack([w[key] for w in weights]), dim=0
        )

    return avg_weights

### 8. Federated Training Loop

In [ ]:
global_model = SimpleNN()

rounds = 10

for r in range(rounds):

    client_weights = []

    for client_data in clients:
        local_model = SimpleNN()
        local_model.load_state_dict(global_model.state_dict())

        weights = train_local(local_model, client_data)

        client_weights.append(weights)

    new_weights = federated_average(client_weights)

    global_model.load_state_dict(new_weights)

    print("Round", r, "completed")

9. Membership Inference Attack

In [ ]:
def get_confidence(model, dataset):

    loader = DataLoader(dataset, batch_size=32)

    model.eval()

    confidences = []

    with torch.no_grad():
        for data, _ in loader:

            output = model(data)
            probs = torch.softmax(output, dim=1)

            max_conf = torch.max(probs, dim=1)[0]

            confidences.extend(max_conf.numpy())

    return confidences

### 10. Attacker Model

In [ ]:
attack_model = LogisticRegression()
attack_model.fit(X_train, y_train)

### 11. Evaluation and Comparison

In [ ]:
print("Centralized Model Attack Accuracy:", central_attack_acc)
print("Federated Model Attack Accuracy:", fed_attack_acc)

plt.bar(["Centralized", "Federated"], [central_attack_acc, fed_attack_acc])
plt.title("Membership Inference Attack Accuracy")
plt.ylabel("Accuracy")
plt.show()

### 12. Discussion